# The artifact tokens

A demo on the high-norm tokens of large ViTs (Darcet et al., *Vision Transformers Need Registers*): tokens
in uninformative patches that the model recycles to store global information. In NAVE they tend to form a
concept of their own, made of the most uncertain tokens.

One reading: if the semantic concepts are compact and well separated, a token that averages them all
points far from each of them, and the average forms a separable concept of its own. This notebook measures,
block by block:
- the **norm** of the artifact tokens against the others;
- the **cosine** between the artifact concept's prototype and the **mean** of the other prototypes (close to 1
  if the artifacts are their average);
- the **separation** of the other concepts (mean cosine between their prototypes: lower is more separated);
- at the last block, **uncertainty against norm**, token by token.

Large, long-trained ViTs show artifacts most clearly: try `("vits16", "dino")` and `("vits16", "imagenet")`.

**Needs:** `pip install "nave-xai[animation]"`. **Downloads:** none with ViT-Ti/16; about 90 MB per ViT-S/16.

In [ ]:
# %pip install "nave-xai[animation]"
import numpy as np

MODELS = [("vitt16", "imagenet"), ("vits16", "imagenet"), ("vits16", "dino")]
K = 8
N_IMAGES = 12
ARTIFACT = None                            # None: the most uncertain concept at the last block
SEED = 0


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display


def match_colours(labels, reference, K):
    """Give every concept of `labels` the colour index of the reference concept it overlaps most
    (one-to-one, maximising the total overlap), so colours stay stable from frame to frame."""
    overlap = np.zeros((K, K))
    np.add.at(overlap, (labels.ravel(), reference.ravel()), 1)
    try:
        from scipy.optimize import linear_sum_assignment
        rows, cols = linear_sum_assignment(-overlap)
        mapping = dict(zip(rows, cols))
    except ImportError:                                     # greedy, without scipy
        mapping, used = {}, set()
        for a, b in sorted(np.ndindex(K, K), key=lambda ab: -overlap[ab]):
            if a not in mapping and b not in used:
                mapping[a] = b
                used.add(b)
    return np.vectorize(lambda a: mapping.get(a, a))(labels)


def knn_agreement(features, targets, k=5, n=1500, seed=0):
    """How well a point's k nearest neighbours (cosine) predict its target: leave-one-out accuracy."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(features), min(n, len(features)), replace=False)
    F = features[idx] / np.linalg.norm(features[idx], axis=1, keepdims=True)
    t = targets[idx]
    sim = F @ F.T
    np.fill_diagonal(sim, -np.inf)
    nearest = np.argsort(-sim, axis=1)[:, :k]
    votes = np.array([np.bincount(t[row], minlength=t.max() + 1).argmax() for row in nearest])
    return float((votes == t).mean())


def normalise_embedding(e):
    e = e - np.median(e, axis=0)
    return e / np.percentile(np.abs(e), 98)


def token_labels(maps, centres):
    """The concept at every patch centre of every image: one label per token."""
    return np.concatenate([m[np.ix_(centres, centres)].ravel() for m in maps])


def match_to(labels, reference, K):
    """Rename `labels` so each concept takes the id of the reference concept it overlaps most."""
    return match_colours(labels, reference, K)


In [ ]:
from skimage import data, transform

def load(name, size=224):
    img = getattr(data, name)()
    img = np.repeat(img[..., None], 3, axis=2) if img.ndim == 2 else img[..., :3]
    return (transform.resize(img, (size, size), anti_aliasing=False) * 255).astype(np.uint8)

names = ["astronaut", "coffee", "chelsea", "rocket", "immunohistochemistry", "hubble_deep_field",
         "retina", "camera", "coins", "horse", "moon", "page"][:N_IMAGES]
images = [load(n) for n in names]

In [ ]:
def block_tokens(backbone, img, names):
    """{name: (196, C) tokens, scaled to length 1} for one 224 px image."""
    mean, std = np.array(backbone.mean, np.float32), np.array(backbone.std, np.float32)
    x = ((img / 255.0 - mean) / std).astype(np.float32).transpose(2, 0, 1)[None]
    out = backbone(x, names)
    res = {}
    for n in names:
        t = out[n][0].reshape(out[n].shape[1], -1).T
        res[n] = t
    return res

centres = np.arange(8, 224, 16)

In [ ]:
from nave_xai import NAVEImage

def artifact_profile(model, training):
    final = NAVEImage(model, training, layers=[12], k=K, seed=SEED).fit(images)
    bb = final.backbone
    exps = final.predict(images, uncertainty=True)
    concept = token_labels([e.explanation for e in exps], centres)
    unc = token_labels([e.uncertainty for e in exps], centres)
    art = ARTIFACT if ARTIFACT is not None else int(np.argmax([unc[concept == c].mean() if np.any(concept == c) else 0
                                                                for c in range(K)]))
    names_b = ["b0"] + [f"b{i}" for i in range(1, 13)]                     # b0: the input of block 1
    per_image = [block_tokens(bb, im, names_b) for im in images]
    out = dict(name=f"{model}, {training}", artifact=art, norm_art=[], norm_other=[], cosine=[], separation=[])
    for n in names_b:
        raw = np.concatenate([p[n] for p in per_image])
        norms = np.linalg.norm(raw, axis=1)
        unit = raw / norms[:, None]
        protos = np.stack([unit[concept == c].mean(0) if np.any(concept == c) else np.zeros(unit.shape[1])
                           for c in range(K)])
        protos /= np.linalg.norm(protos, axis=1, keepdims=True) + 1e-12
        others = np.array([c for c in range(K) if c != art and np.any(concept == c)])
        mean_other = protos[others].mean(0); mean_other /= np.linalg.norm(mean_other) + 1e-12
        cos_pairs = protos[others] @ protos[others].T
        out["norm_art"].append(norms[concept == art].mean()); out["norm_other"].append(norms[concept != art].mean())
        out["cosine"].append(float(protos[art] @ mean_other))
        out["separation"].append(float(cos_pairs[np.triu_indices(len(others), 1)].mean()))
    out["last_norm"], out["last_unc"], out["is_art"] = norms, unc, concept == art
    return out

profiles = [artifact_profile(m, t) for m, t in MODELS]
for p in profiles:
    print(f"{p['name']:18s} artifact concept {p['artifact']}")

In [ ]:
fig, axes = plt.subplots(len(profiles), 4, figsize=(17, 3.3 * len(profiles)), squeeze=False)
b = range(13)                                                             # 0: the input of block 1
for row, p in zip(axes, profiles):
    row[0].plot(b, p["norm_art"], label="artifact tokens"); row[0].plot(b, p["norm_other"], label="others")
    row[0].set_ylabel(p["name"], fontsize=9); row[0].set_title("mean norm", fontsize=9); row[0].legend(fontsize=7)
    row[1].plot(b, p["cosine"], color="C3"); row[1].set_ylim(-1, 1)
    row[1].set_title("cosine(artifact prototype, mean of the others)", fontsize=9)
    row[2].plot(b, p["separation"], color="C2"); row[2].set_ylim(-1, 1)
    row[2].set_title("separation of the others (mean cosine)", fontsize=9)
    row[3].scatter(p["last_norm"][~p["is_art"]], p["last_unc"][~p["is_art"]], s=3, c="0.6", label="others")
    row[3].scatter(p["last_norm"][p["is_art"]], p["last_unc"][p["is_art"]], s=5, c="C3", label="artifact")
    row[3].set_xlabel("norm (last block)", fontsize=8); row[3].set_ylabel("uncertainty", fontsize=8)
    row[3].set_title("uncertainty against norm", fontsize=9); row[3].legend(fontsize=7)
    for ax in row[:3]:
        ax.set_xlabel("block (0: input of block 1)", fontsize=8)
fig.tight_layout()

## Things to look at

- whether the cosine with the others' mean rises as their separation improves;
- whether the artifact tokens' norm grows before or after they form their concept;
- the models side by side: which have artifacts, from which block;
- `K`: with more concepts, do the artifacts stay one concept?

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  editor    = {Ceci, Michelangelo and Kocev, Dragi and Monreale, Anna and Vens, Celine and Flach, Peter
               and Manco, Giuseppe and Spiliopoulou, Myra and Masciari, Elio},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26},
  isbn      = {978-3-032-37664-0}
}
```